# 00. Model and Agent Basics

**Тема 7: Retrieval-Augmented Generation**  
Курс «Промышленная разработка AI-агентов» · ФКН ВШЭ

---

Этот вводный ноутбук нужен перед `01-04`.

Здесь мы не строим RAG. Мы проверяем базовую инфраструктуру:
- подключение к chat model GigaChat;
- подключение к embeddings model GigaChat;
- минимальный agent loop на LangChain;
- минимальный Deep Agents harness;
- Phoenix tracing для LLM / agent вызовов.


## Что должно получиться к концу ноутбука

К концу ноутбука студент должен уметь:
- объяснить разницу между chat model и embeddings model;
- проверить, что `.env` и credentials работают;
- вызвать `llm.invoke(...)` напрямую;
- получить embedding vectors и посчитать простое сходство;
- собрать минимального агента через `create_agent()`;
- собрать минимального Deep Agent через `create_deep_agent()`;
- открыть Phoenix и найти там trace прямого вызова модели и agent run.


## Outline

1. Setup и `.env`
2. Phoenix tracing
3. GigaChat chat model
4. GigaChat embeddings
5. Минимальный LangChain agent
6. Минимальный Deep Agent
7. Что дальше


In [ ]:
# Установка зависимостей (запустить один раз в выбранном Jupyter kernel)
# %pip install -r ../requirements.txt

# После установки зависимостей перезапустите kernel: Kernel -> Restart.


In [1]:
from __future__ import annotations

import math
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langchain_gigachat import GigaChat, GigaChatEmbeddings
from langgraph.checkpoint.memory import MemorySaver


## 1. Setup и `.env`

Все notebook-и в теме используют один и тот же `.env` в корне `rag_course_repo`.

Минимально нужны:

```env
GIGACHAT_CREDENTIALS=<Base64(client_id:client_secret)>
GIGACHAT_SCOPE=GIGACHAT_API_PERS
GIGACHAT_MODEL=GigaChat-2-Max
GIGACHAT_EMBEDDINGS_MODEL=EmbeddingsGigaR
```

`verify_ssl_certs=False` в учебных ноутбуках оставлен для локальных окружений, где часто мешают корпоративные сертификаты. В production такой default нужно пересмотреть.


In [2]:
repo_root_candidates = [Path.cwd(), Path.cwd().parent]
REPO_ROOT = next(
    (path for path in repo_root_candidates if (path / '.env.example').exists()),
    Path.cwd(),
)
TOPIC7_ROOT = next(
    (
        path for path in [Path.cwd(), Path.cwd() / 'topic7_rag', REPO_ROOT / 'topic7_rag']
        if path.exists() and path.name == 'topic7_rag'
    ),
    REPO_ROOT / 'topic7_rag',
)
ENV_PATH = REPO_ROOT / '.env'

if ENV_PATH.exists():
    load_dotenv(ENV_PATH)
    env_status = str(ENV_PATH)
else:
    env_status = f'не найден; скопируйте {REPO_ROOT / ".env.example"} -> {ENV_PATH}'

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
GIGACHAT_SCOPE = os.getenv('GIGACHAT_SCOPE', 'GIGACHAT_API_PERS')
GIGACHAT_MODEL = os.getenv('GIGACHAT_MODEL', 'GigaChat-2-Max')
GIGACHAT_EMBEDDINGS = os.getenv('GIGACHAT_EMBEDDINGS_MODEL', 'EmbeddingsGigaR')
GIGACHAT_TIMEOUT = float(os.getenv('GIGACHAT_TIMEOUT', '60'))

PHOENIX_PROJECT_NAME = os.getenv('PHOENIX_PROJECT_NAME_00', 'topic7-rag-00')
PHOENIX_COLLECTOR_ENDPOINT = os.getenv('PHOENIX_COLLECTOR_ENDPOINT')
PHOENIX_WORKING_DIR = os.getenv('PHOENIX_WORKING_DIR', str(REPO_ROOT / '.phoenix'))
PHOENIX_PORT = int(os.getenv('PHOENIX_PORT', '6006'))
PHOENIX_GRPC_PORT = int(os.getenv('PHOENIX_GRPC_PORT', '4317'))
PHOENIX_ENABLED = False
PHOENIX_SESSION = None
PHOENIX_SESSION_URL = None

print('✓ Окружение загружено')
print(f'  Env file:          {env_status}')
print(f'  Repo root:         {REPO_ROOT}')
print(f'  Topic7 root:       {TOPIC7_ROOT}')
print(f'  GigaChat model:    {GIGACHAT_MODEL}')
print(f'  Embeddings model:  {GIGACHAT_EMBEDDINGS}')
print(f'  Phoenix project:   {PHOENIX_PROJECT_NAME}')


✓ Окружение загружено
  Env file:          /Users/Sergej/claude_code/agent_course/rag_course_repo/.env
  Repo root:         /Users/Sergej/claude_code/agent_course/rag_course_repo
  Topic7 root:       /Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag
  GigaChat model:    GigaChat-2-Max
  Embeddings model:  EmbeddingsGigaR
  Phoenix project:   topic7-rag-00


In [3]:
if not GIGACHAT_CREDENTIALS or GIGACHAT_CREDENTIALS == 'your_base64_credentials_here':
    raise ValueError(
        'Не найден GIGACHAT_CREDENTIALS. Скопируйте .env.example в .env и заполните credentials.'
    )

print('✓ GIGACHAT_CREDENTIALS найден')


✓ GIGACHAT_CREDENTIALS найден


## 2. Phoenix tracing

Phoenix нужен, чтобы увидеть agent run не как магию, а как последовательность событий:
- прямой LLM call;
- embedding call;
- agent loop;
- tool calls.

Поддерживаются два режима:
- внешний collector через `PHOENIX_COLLECTOR_ENDPOINT`;
- локальный Phoenix на `PHOENIX_PORT`.


In [4]:
try:
    import socket
    from urllib.error import HTTPError, URLError
    from urllib.request import urlopen

    import phoenix as px
    from phoenix.otel import register
    from openinference.instrumentation.langchain import LangChainInstrumentor

    def is_port_open(host: str, port: int, timeout: float = 0.5) -> bool:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
            sock.settimeout(timeout)
            return sock.connect_ex((host, port)) == 0

    def check_phoenix_ui(url: str) -> None:
        try:
            with urlopen(url, timeout=2) as response:
                print(f'Phoenix UI отвечает: HTTP {response.status}')
        except HTTPError as exc:
            print(f'Phoenix UI вернул HTTP {exc.code}; tracing endpoint всё равно может работать.')
        except URLError as exc:
            print(f'Phoenix UI пока не ответил ({type(exc.reason).__name__}); попробуйте обновить страницу через несколько секунд.')
        except Exception as exc:
            print(f'Phoenix UI check пропущен: {type(exc).__name__}: {exc}')

    if PHOENIX_COLLECTOR_ENDPOINT:
        collector_endpoint = PHOENIX_COLLECTOR_ENDPOINT.rstrip('/')
        if not collector_endpoint.endswith('/v1/traces'):
            collector_endpoint = collector_endpoint + '/v1/traces'
        PHOENIX_SESSION_URL = None
        print(f'Используем внешний Phoenix collector: {collector_endpoint}')
    else:
        local_ui_url = f'http://localhost:{PHOENIX_PORT}/'
        collector_endpoint = f'http://127.0.0.1:{PHOENIX_PORT}/v1/traces'

        if is_port_open('127.0.0.1', PHOENIX_PORT):
            PHOENIX_SESSION_URL = local_ui_url
            print(f'Используем уже запущенный локальный Phoenix: {PHOENIX_SESSION_URL}')
            check_phoenix_ui(PHOENIX_SESSION_URL)
        else:
            if is_port_open('127.0.0.1', PHOENIX_GRPC_PORT):
                raise RuntimeError(
                    f'gRPC port {PHOENIX_GRPC_PORT} уже занят, а HTTP port {PHOENIX_PORT} свободен. '
                    'Перезапустите kernel или поменяйте PHOENIX_PORT/PHOENIX_GRPC_PORT в .env.'
                )
            os.environ['PHOENIX_PORT'] = str(PHOENIX_PORT)
            os.environ['PHOENIX_GRPC_PORT'] = str(PHOENIX_GRPC_PORT)
            os.environ['PHOENIX_WORKING_DIR'] = PHOENIX_WORKING_DIR
            Path(PHOENIX_WORKING_DIR).mkdir(parents=True, exist_ok=True)
            PHOENIX_SESSION = px.launch_app(use_temp_dir=False)
            if PHOENIX_SESSION is None:
                raise RuntimeError('Phoenix launch_app() не смог поднять локальную сессию')
            PHOENIX_SESSION_URL = PHOENIX_SESSION.url
            collector_endpoint = f'http://127.0.0.1:{PHOENIX_SESSION.port}/v1/traces'
            print(f'Локальный Phoenix запущен: {PHOENIX_SESSION_URL}')
            print(f'Phoenix DB dir: {PHOENIX_WORKING_DIR}')

    tracer_provider = register(
        endpoint=collector_endpoint,
        protocol='http/protobuf',
        project_name=PHOENIX_PROJECT_NAME,
        batch=False,
        auto_instrument=False,
    )
    LangChainInstrumentor().instrument(tracer_provider=tracer_provider)
    PHOENIX_ENABLED = True
    print(f'✓ Phoenix tracing включен: {collector_endpoint}')
except Exception as exc:
    print('Phoenix tracing пропущен:', type(exc).__name__, exc)
    print('Если нужен tracing, проверьте kernel, занятые порты и endpoint collector-а.')


/Users/Sergej/claude_code/agent_course/rag_course_repo/.venv/lib/python3.12/site-packages/authlib/_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


🌍 To view the Phoenix app in your browser, visit http://localhost:6006/
💽 Your data is being persisted to sqlite:////Users/Sergej/claude_code/agent_course/rag_course_repo/topic7_rag/.phoenix/phoenix.db
📖 For more information on how to use Phoenix, check out https://arize.com/docs/phoenix
Локальный Phoenix запущен: http://localhost:6006/
Phoenix DB dir: ./.phoenix
🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: topic7-rag-00
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://127.0.0.1:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.

✓ Phoenix tracing включен: http://127.0.0.1:6006/v1/traces


## 3. Chat model: прямой вызов GigaChat

Chat model принимает сообщения и возвращает сообщение.

Это базовый building block для всех следующих уровней:
- single LLM call;
- LangChain agent;
- Deep Agent;
- RAG generation step.


In [5]:
llm = GigaChat(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=GIGACHAT_MODEL,
    temperature=0,
    timeout=GIGACHAT_TIMEOUT,
    verify_ssl_certs=False,
)

response = llm.invoke([
    SystemMessage(content='Отвечай кратко и по делу.'),
    HumanMessage(content='В одном предложении объясни, зачем RAG нужен агентам.'),
])

print(response.content if hasattr(response, 'content') else response)


RAG (Retrieval-Augmented Generation) помогает агентам генерировать более точные и информативные ответы за счет использования внешних источников данных в дополнение к собственным знаниям модели.


## 4. Embeddings: векторное представление текста

Embeddings model превращает текст в числовой вектор.

В RAG это нужно для dense retrieval: похожие по смыслу фрагменты должны иметь близкие векторы.

Ниже — маленький sanity check без Chroma: получим векторы и посчитаем cosine similarity вручную.


In [6]:
embeddings = GigaChatEmbeddings(
    credentials=GIGACHAT_CREDENTIALS,
    scope=GIGACHAT_SCOPE,
    model=GIGACHAT_EMBEDDINGS,
    timeout=GIGACHAT_TIMEOUT,
    verify_ssl_certs=False,
)

texts = [
    'RAG ищет релевантные чанки в базе знаний и добавляет их в prompt.',
    'GraphRAG строит граф сущностей и связей поверх документов.',
    'Файловая память помогает агенту помнить устойчивые выводы между запусками.',
]
query = 'Как агент находит подходящие фрагменты документов?'

doc_vectors = embeddings.embed_documents(texts)
query_vector = embeddings.embed_query(query)

print(f'Doc vectors: {len(doc_vectors)}')
print(f'Vector dimension: {len(doc_vectors[0])}')
print(f'Query vector dimension: {len(query_vector)}')


Doc vectors: 3
Vector dimension: 2560
Query vector dimension: 2560


In [7]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    norm_a = math.sqrt(sum(x * x for x in a))
    norm_b = math.sqrt(sum(y * y for y in b))
    return dot / (norm_a * norm_b)

scores = [cosine_similarity(query_vector, vec) for vec in doc_vectors]
ranked = sorted(zip(texts, scores), key=lambda item: item[1], reverse=True)

for text, score in ranked:
    print(f'{score:.3f}  {text}')


0.682  GraphRAG строит граф сущностей и связей поверх документов.
0.679  RAG ищет релевантные чанки в базе знаний и добавляет их в prompt.
0.650  Файловая память помогает агенту помнить устойчивые выводы между запусками.


## 5. Минимальный LangChain agent

`create_agent()` дает простой agent loop:

```text
model -> maybe tool call -> observation -> model -> answer
```

Для первого примера сделаем два игрушечных tools. Они не заменяют RAG, но показывают механику tool calling.


In [8]:
@tool
def course_glossary(term: str) -> str:
    """Возвращает короткое объяснение термина из темы RAG."""
    glossary = {
        'rag': 'Retrieval-Augmented Generation: поиск внешнего контекста + генерация ответа по этому контексту.',
        'chunk': 'Фрагмент документа, который индексируется и возвращается retrieval-слоем.',
        'embedding': 'Числовой вектор текста, используемый для поиска по смысловой близости.',
        'graphrag': 'RAG-подход, где знания представлены через сущности и связи в графе.',
    }
    key = term.strip().lower()
    return glossary.get(key, f'Термин {term!r} пока не добавлен в мини-глоссарий.')


@tool
def compare_numbers(a: float, b: float) -> str:
    """Сравнивает два числа и возвращает большее."""
    if a == b:
        return f'Числа равны: {a}'
    return f'Большее число: {max(a, b)}'


langchain_agent = create_agent(
    model=llm,
    tools=[course_glossary, compare_numbers],
    system_prompt=(
        'Ты учебный агент для курса про RAG. '
        'Если вопрос касается терминов, используй course_glossary. '
        'Отвечай кратко.'
    ),
)

print('✓ LangChain agent создан')


✓ LangChain agent создан


In [9]:
agent_result = langchain_agent.invoke({
    'messages': [
        {
            'role': 'user',
            'content': 'Объясни термин embedding и скажи, почему он важен для RAG.',
        }
    ]
})

for message in agent_result['messages']:
    message.pretty_print()


================================ Human Message =================================

Объясни термин embedding и скажи, почему он важен для RAG.
================================== Ai Message ==================================
Tool Calls:
  course_glossary (297acf7e-24ee-48d9-b461-115f207116ba)
 Call ID: 297acf7e-24ee-48d9-b461-115f207116ba
  Args:
    term: embedding
================================= Tool Message =================================
Name: course_glossary

Числовой вектор текста, используемый для поиска по смысловой близости.
================================== Ai Message ==================================

Embedding — это числовой вектор текста, который позволяет искать информацию не только по точному совпадению слов, но и по смыслу. Это важно в RAG, так как обеспечивает более точное извлечение релевантных данных из больших объемов информации.


## 6. Минимальный Deep Agent

Deep Agents — более высокий слой поверх LangChain / LangGraph.

Для нас сейчас важно не всё API, а идея harness-а:
- planning и task management встроены;
- filesystem / memory / subagents можно подключать как middleware/backend;
- тот же `llm` и tools можно переиспользовать.

В этом вводном ноутбуке делаем простой Deep Agent без файловой памяти. В `04_file_memory.ipynb` вернемся к filesystem и persistence подробнее.


In [10]:
try:
    from deepagents import create_deep_agent

    deep_agent = create_deep_agent(
        model=llm,
        tools=[course_glossary],
        checkpointer=MemorySaver(),
        system_prompt=(
            'Ты учебный Deep Agent для курса про RAG. '
            'Планируй коротко, используй tools только если это помогает, '
            'и отвечай на русском.'
        ),
    )
    print('✓ Deep Agent создан')
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'Пакет deepagents не установлен. Установите зависимости: python -m pip install -r ../requirements.txt'
    ) from exc


✓ Deep Agent создан


In [11]:
deep_config = {'configurable': {'thread_id': 'topic7-basics-deep-agent'}}

deep_result = deep_agent.invoke(
    {
        'messages': [
            {
                'role': 'user',
                'content': 'Кратко объясни, что такое RAG, и почему простого prompt-а иногда недостаточно.',
            }
        ]
    },
    config=deep_config,
)

for message in deep_result['messages']:
    message.pretty_print()


================================ Human Message =================================

Кратко объясни, что такое RAG, и почему простого prompt-а иногда недостаточно.
================================== Ai Message ==================================

RAG (Retrieval-Augmented Generation) — подход к генерации текста с использованием предварительно извлеченной информации из внешних источников данных. Простого prompt-а может быть недостаточно, так как он ограничен контекстом запроса пользователя и не всегда содержит всю необходимую информацию или детали для создания качественного ответа.


## 7. Phoenix status

После прямого вызова модели, embeddings и agent runs откройте Phoenix UI и найдите project из этого ноутбука.


In [12]:
print('Phoenix enabled:', PHOENIX_ENABLED)
if PHOENIX_ENABLED:
    print('Phoenix project:', PHOENIX_PROJECT_NAME)
    print('Phoenix working dir:', PHOENIX_WORKING_DIR)
    print('Collector endpoint:', PHOENIX_COLLECTOR_ENDPOINT or f'local -> http://127.0.0.1:{PHOENIX_PORT}/v1/traces')
    if PHOENIX_SESSION_URL:
        print('Phoenix UI:', PHOENIX_SESSION_URL)
else:
    print('Phoenix не активирован в текущем окружении.')


Phoenix enabled: True
Phoenix project: topic7-rag-00
Phoenix working dir: ./.phoenix
Collector endpoint: local -> http://127.0.0.1:6006/v1/traces
Phoenix UI: http://localhost:6006/


## Что важно вынести

- Chat model отвечает на сообщения; embeddings model превращает текст в вектор.
- LangChain `create_agent()` — минимальный способ получить tool-using agent loop.
- Deep Agents — более высокий harness для задач, где скоро понадобятся planning, files, memory и subagents.
- Phoenix полезен уже на этом уровне: он показывает, где был обычный LLM call, где embeddings, а где agent/tool loop.

Дальше:
- `01_ingestion_pipeline.ipynb` — построим базу знаний;
- `02_retrieval_generation.ipynb` — соберем RAG и agentic RAG;
- `03_graphrag.ipynb` — перейдем к графу сущностей;
- `04_file_memory.ipynb` — добавим workspace memory.


## Exercises

1. Добавь еще 2 термина в `course_glossary`, например `reranking` и `citation`.
2. Измени prompt LangChain agent так, чтобы он всегда отвечал с одним примером.
3. Сравни trace прямого `llm.invoke(...)` и trace `langchain_agent.invoke(...)` в Phoenix.
4. Запусти Deep Agent два раза с тем же `thread_id` и посмотри, что меняется в trace.
5. Подумай, где в следующих ноутбуках нужен chat model, а где embeddings model.
